In [ ]:
import json
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import matplotlib.pyplot as plt


LOG_PATH = Path("/home/hadoop-intelligence-studio/dolphinfs_ssd_hadoop-intelligence-studio/songbaijun/DeepWorld/exps/demo/logs.jsonl")
METRIC_NAMES = ["loss"]
METRIC_EMA_WINDOW = 1
FIGURE_SIZE = (10, 4)


def should_include_entry(entry: dict) -> bool:
	return entry.get("event") == "train"


def load(path: Path) -> list[dict]:
	assert path.exists()
	entries: list[dict] = []
	with path.open("r", encoding="utf-8") as handle:
		lines = [line.strip() for line in handle if line.strip()]
		for line in lines:
			try: entries.append(json.loads(line))
			except json.JSONDecodeError as error:
				continue
	return entries


def filter(
	entries: Iterable[dict],
	condition:  Callable[[dict], bool]
) -> list[dict]:
	return [entry for entry in entries if condition[entry]]


def smooth(values: np.ndarray) -> np.ndarray:
	window = min(METRIC_EMA_WINDOW, len(values))
	if window <= 1:
		return values
	cumulative = np.concatenate([np.array([0.0]), values.cumsum()])
	starts = np.maximum(np.arange(len(values)) + 1 - window, 0)
	sums = cumulative[1:] - cumulative[starts]
	counts = np.minimum(np.arange(len(values)) + 1, window)
	return sums / counts


def plot_metric(entries: list[dict], metric_name: str) -> None:
	entries = filter(entries, lambda e: "step" in e and metric_name in e)
	entries = filter(entries, should_include_entry)
	steps = np.array([e["step"] for e in entries])
	values = np.array([e[metric_name] for e in entries])

	fig, axis = plt.subplots(1, 1, figsize=FIGURE_SIZE)
	if len(steps) == 0:
		axis.set_title(f"{metric_name}: no matching entries")
	else:
		axis.plot(steps, smooth(values), linewidth=1.5)
		axis.set_title(metric_name)
	axis.set_xlabel("step")
	axis.set_ylabel(metric_name)
	axis.grid(True, alpha=0.3)
	fig.tight_layout()
	fig.show()


def main() -> None:
	entries = load(LOG_PATH)
	for metric_name in METRIC_NAMES:
		plot_metric(entries, metric_name)


if __name__ == "__main__":
	main()
